# SQL Analysis: Lithuanian vs Mexican CIS Cohorts

This notebook loads both cleaned datasets into a SQLite database and runs six comparative SQL queries covering conversion rates, demographics, biomarkers (measurable biological indicators such as oligoclonal bands and IgG levels in spinal fluid), MRI findings, and Visual Evoked Potentials (VEP) and Brainstem Auditory Evoked Potentials (BAEP) - tests that measure how quickly electrical signals travel along the visual and auditory nerve pathways respectively. Query 6 produces a cross-population union exported for use in Power BI.

An SQLite database is a file that stores data in structured tables to be queried with SQL. Unlike a spreadsheet, a database has one or more tables, understands relationships between them, and can answer queries across them efficiently. SQLite was the selected tool for this project because it only has 2 small tables.

## 1. Create SQLite Database and Load Tables

Load both cleaned CSVs with pandas and write them to a SQLite database at `sql/ms_cis.db`. Each cohort becomes its own table (`lithuanian`, `mexican`). The messy mixed-label strings in the Mexican dataset are preserved as-is; the SQL queries normalise them at query time using `SUBSTR(TRIM(...), 1, 1)` to extract the leading digit.

In [1]:
import sqlite3
import pandas as pd
import os

lt = pd.read_csv('../data/cleaned/lithuanian_cleaned.csv')
mx = pd.read_csv('../data/cleaned/mexican_cleaned.csv')

os.makedirs('../sql', exist_ok=True)
conn = sqlite3.connect('../sql/ms_cis.db')
lt.to_sql('lithuanian', conn, if_exists='replace', index=False)
mx.to_sql('mexican',    conn, if_exists='replace', index=False)

print('Database: sql/ms_cis.db')
print(f'  lithuanian : {len(lt)} rows x {len(lt.columns)} columns')
print(f'  mexican    : {len(mx)} rows x {len(mx.columns)} columns')

Database: sql/ms_cis.db
  lithuanian : 138 rows x 44 columns
  mexican    : 288 rows x 18 columns


## Query 1: Conversion Counts and Rate per Cohort

Query in plain English: of all the patients with a recorded outcome, how many converted to `MS` and how many did not, and what percentage do these groups represent?

The finding shows that Lithuanian patients converted at *35.5%* and Mexican patients at *45.8%* - a 10-percentage-point difference between the two cohorts, which could reflect differences in follow-up duration or population genetics. Published CIS literature tends to put conversion rates between 30% and 50% over a 2-5 year follow-up. Both cohorts in this study fall within that range.

 • **CAST** converts data from one type to another. In this instance, the `MS` and `group` columns are made up of *1s* and 0s in string format, so this converts them to actual numbers for mathematical querying.

 • **CASE WHEN ... THEN ... ELSE ... END AS** is SQL's version of an if/else statement. It reads: IF the converted `MS` value = *1*, THEN label it *Converter*. ELSE, label it *Non-converter*. **END AS** `outcome` closes the statement and names this newly created column `outcome`.

 • **GROUP BY** bundles the patients into two groups (*1s* and *non-1s*), mirroring the same conversion logic used with **CAST**.

 • **TRIM** removes any spaces at the beginning or end of the text. **SUBSTR** shortens the text from an indexed position - for example, *1, 1* returns the first character only.

 • **UNION ALL** takes the output table from the Lithuanian query and stacks the Mexican query directly underneath it. This only works when both queries return the exact same number of columns - in this step, both sides return exactly *4*: `cohort`, `outcome`, `n`, and `conversion_rate_pct`.

In [2]:
q1 = """
SELECT 'Lithuanian' AS cohort,
       CASE WHEN CAST(MS AS INTEGER) = 1 THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 /
             (SELECT COUNT(*) FROM lithuanian WHERE MS IS NOT NULL), 1) AS conversion_rate_pct
FROM lithuanian
WHERE MS IS NOT NULL
GROUP BY CAST(MS AS INTEGER)
UNION ALL
SELECT 'Mexican' AS cohort,
       CASE WHEN SUBSTR(TRIM("group"), 1, 1) = '1' THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 /
             (SELECT COUNT(*) FROM mexican WHERE "group" IS NOT NULL), 1) AS conversion_rate_pct
FROM mexican
WHERE "group" IS NOT NULL
GROUP BY SUBSTR(TRIM("group"), 1, 1)
ORDER BY cohort, outcome
"""
df_q1 = pd.read_sql(q1, conn)
display(df_q1)

,cohort,outcome,n,conversion_rate_pct
0,Lithuanian,Converter,49,35.5
1,Lithuanian,Non-converter,89,64.5
2,Mexican,Converter,126,45.8
3,Mexican,Non-converter,149,54.2


## Query 2: Average Age by Conversion Outcome per Cohort

Query in plain English: Were converters older or younger than non-converters at the time of the CIS episode, in each cohort?

The query found that converters were slightly older than non-converters in the Mexican dataset, and slightly younger in the Lithuanian dataset.

These age differences are small, and the fact that each cohort points in opposite directions suggests that age alone is not a strong or consistent predictor of conversion. Negative findings are useful too - they outline variables that fail to predict conversion, which is as informative as identifying variables that do.

In [3]:
q2 = """
SELECT 'Lithuanian' AS cohort,
       CASE WHEN CAST(MS AS INTEGER) = 1 THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       ROUND(AVG(Age), 1) AS avg_age
FROM lithuanian
WHERE MS IS NOT NULL
GROUP BY CAST(MS AS INTEGER)
UNION ALL
SELECT 'Mexican' AS cohort,
       CASE WHEN SUBSTR(TRIM("group"), 1, 1) = '1' THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       ROUND(AVG("Age (y)"), 1) AS avg_age
FROM mexican
WHERE "group" IS NOT NULL
GROUP BY SUBSTR(TRIM("group"), 1, 1)
ORDER BY cohort, outcome
"""
df_q2 = pd.read_sql(q2, conn)
display(df_q2)

,cohort,outcome,avg_age
0,Lithuanian,Converter,39.9
1,Lithuanian,Non-converter,41.8
2,Mexican,Converter,34.8
3,Mexican,Non-converter,33.4


## Query 3: Oligoclonal Band Positivity Rate by Conversion Outcome

Query in plain English: of the patients who had a positive OGB test result, what proportion went on to convert across both cohorts?

The query found that *71%* of Lithuanian and *47%* of Mexican converters tested OGB positive. Non-converters had OGB positivity of only *15%* for Lithuania and *13%* for Mexico.

This is the strongest finding that holds consistently across both cohorts, strongly indicating that immune activity inside the central nervous system is associated with conversion regardless of which country the patient is from. This validates the finding from Daniel et al. 2024 using independently written queries on the same datasets.

In [4]:
q3 = """
SELECT 'Lithuanian' AS cohort,
       CASE WHEN CAST(MS AS INTEGER) = 1 THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       SUM(CASE WHEN TRIM("OGB + in CSF") = '1' THEN 1 ELSE 0 END) AS ogb_positive,
       COUNT(CASE WHEN TRIM("OGB + in CSF") IN ('0', '1') THEN 1 END) AS ogb_tested,
       ROUND(
           SUM(CASE WHEN TRIM("OGB + in CSF") = '1' THEN 1 ELSE 0 END) * 100.0 /
           NULLIF(COUNT(CASE WHEN TRIM("OGB + in CSF") IN ('0', '1') THEN 1 END), 0),
       1) AS ogb_positive_pct
FROM lithuanian
WHERE MS IS NOT NULL
GROUP BY CAST(MS AS INTEGER)
UNION ALL
SELECT 'Mexican' AS cohort,
       CASE WHEN SUBSTR(TRIM("group"), 1, 1) = '1' THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       SUM(CASE WHEN SUBSTR(TRIM("Oligoclonal bands"), 1, 1) = '1' THEN 1 ELSE 0 END) AS ogb_positive,
       COUNT(CASE WHEN SUBSTR(TRIM("Oligoclonal bands"), 1, 1) IN ('0', '1') THEN 1 END) AS ogb_tested,
       ROUND(
           SUM(CASE WHEN SUBSTR(TRIM("Oligoclonal bands"), 1, 1) = '1' THEN 1 ELSE 0 END) * 100.0 /
           NULLIF(COUNT(CASE WHEN SUBSTR(TRIM("Oligoclonal bands"), 1, 1) IN ('0', '1') THEN 1 END), 0),
       1) AS ogb_positive_pct
FROM mexican
WHERE "group" IS NOT NULL
GROUP BY SUBSTR(TRIM("group"), 1, 1)
ORDER BY cohort, outcome
"""
df_q3 = pd.read_sql(q3, conn)
display(df_q3)

,cohort,outcome,ogb_positive,ogb_tested,ogb_positive_pct
0,Lithuanian,Converter,30,42,71.4
1,Lithuanian,Non-converter,10,66,15.2
2,Mexican,Converter,59,126,46.8
3,Mexican,Non-converter,18,138,13.0


## Query 4: MRI Lesion Presence Rates by Conversion Outcome

Query in plain English: were `periventricular`, `infratentorial`, and `spinal cord` MRI lesions more common in converters than non-converters, across both cohorts?

This query produced the largest single numerical gap in the entire analysis. *79%* of Mexican converters had `periventricular` lesions present, compared to only *26%* of non-converters - a *53*-percentage-point difference. Breakdown by group:

Converters group (*126* Mexican patients): *79%* had `periventricular` lesions present. *21%* did not.

Non-converters group (*149* Mexican patients): *26%* had `periventricular` lesions present. *74%* did not.

`Periventricular` lesions are the most diagnostically significant MRI finding in MS and are specifically named in the McDonald Criteria as characteristic lesion locations. However, this dramatic separation only appears clearly in the Mexican cohort - the Lithuanian cohort showed less MRI separation, likely reflecting the Lithuanian study's greater emphasis on neurological examination findings over imaging.

In [5]:
q4 = """
SELECT 'Lithuanian' AS cohort,
       CASE WHEN CAST(MS AS INTEGER) = 1 THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       ROUND(SUM(CASE WHEN TRIM(Periventricular) = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN TRIM(Periventricular) IN ('0','1') THEN 1 END), 0), 1) AS periventricular_pct,
       ROUND(SUM(CASE WHEN TRIM("MRI lesion localisation: infratentorally") = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN TRIM("MRI lesion localisation: infratentorally") IN ('0','1') THEN 1 END), 0), 1) AS infratentorial_pct,
       ROUND(SUM(CASE WHEN TRIM("MRI spinal lesions") = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN TRIM("MRI spinal lesions") IN ('0','1') THEN 1 END), 0), 1) AS spinal_cord_pct
FROM lithuanian
WHERE MS IS NOT NULL
GROUP BY CAST(MS AS INTEGER)
UNION ALL
SELECT 'Mexican' AS cohort,
       CASE WHEN SUBSTR(TRIM("group"), 1, 1) = '1' THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       ROUND(SUM(CASE WHEN SUBSTR(TRIM("Periventricular MRI"), 1, 1) = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN SUBSTR(TRIM("Periventricular MRI"), 1, 1) IN ('0','1') THEN 1 END), 0), 1) AS periventricular_pct,
       ROUND(SUM(CASE WHEN SUBSTR(TRIM("Infratentorial MRI"), 1, 1) = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN SUBSTR(TRIM("Infratentorial MRI"), 1, 1) IN ('0','1') THEN 1 END), 0), 1) AS infratentorial_pct,
       ROUND(SUM(CASE WHEN SUBSTR(TRIM("Spinal cord MRI"), 1, 1) = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN SUBSTR(TRIM("Spinal cord MRI"), 1, 1) IN ('0','1') THEN 1 END), 0), 1) AS spinal_cord_pct
FROM mexican
WHERE "group" IS NOT NULL
GROUP BY SUBSTR(TRIM("group"), 1, 1)
ORDER BY cohort, outcome
"""
df_q4 = pd.read_sql(q4, conn)
display(df_q4)

,cohort,outcome,periventricular_pct,infratentorial_pct,spinal_cord_pct
0,Lithuanian,Converter,42.2,40.0,80.0
1,Lithuanian,Non-converter,42.9,36.4,41.4
2,Mexican,Converter,79.4,50.0,37.3
3,Mexican,Non-converter,26.2,12.1,26.8


## Query 5: VEP and BAEP Positivity Rates by Conversion Outcome

Visual evoked potentials (VEP) and brainstem auditory evoked potentials (BAEP) are neurophysiological markers of demyelination. Lithuanian columns: `VEP +`, `BAEP +`. Mexican columns: `VEP`, `BAEP`. A positive result (1) indicates an abnormal evoked potential.

In [6]:
q5 = """
SELECT 'Lithuanian' AS cohort,
       CASE WHEN CAST(MS AS INTEGER) = 1 THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       ROUND(SUM(CASE WHEN TRIM("VEP +") = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN TRIM("VEP +") IN ('0','1') THEN 1 END), 0), 1) AS vep_positive_pct,
       ROUND(SUM(CASE WHEN TRIM("BAEP +") = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN TRIM("BAEP +") IN ('0','1') THEN 1 END), 0), 1) AS baep_positive_pct
FROM lithuanian
WHERE MS IS NOT NULL
GROUP BY CAST(MS AS INTEGER)
UNION ALL
SELECT 'Mexican' AS cohort,
       CASE WHEN SUBSTR(TRIM("group"), 1, 1) = '1' THEN 'Converter' ELSE 'Non-converter' END AS outcome,
       ROUND(SUM(CASE WHEN SUBSTR(TRIM(VEP), 1, 1) = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN SUBSTR(TRIM(VEP), 1, 1) IN ('0','1') THEN 1 END), 0), 1) AS vep_positive_pct,
       ROUND(SUM(CASE WHEN SUBSTR(TRIM(BAEP), 1, 1) = '1' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(CASE WHEN SUBSTR(TRIM(BAEP), 1, 1) IN ('0','1') THEN 1 END), 0), 1) AS baep_positive_pct
FROM mexican
WHERE "group" IS NOT NULL
GROUP BY SUBSTR(TRIM("group"), 1, 1)
ORDER BY cohort, outcome
"""
df_q5 = pd.read_sql(q5, conn)
display(df_q5)

,cohort,outcome,vep_positive_pct,baep_positive_pct
0,Lithuanian,Converter,51.3,48.6
1,Lithuanian,Non-converter,36.8,17.6
2,Mexican,Converter,41.3,7.9
3,Mexican,Non-converter,22.1,6.0


## Query 6: Cross-Population Union of Shared Variables

Combine both cohorts into a single flat table containing only the variables present in both datasets, normalised to consistent integer encodings (0/1). A `population` column labels each row as Lithuanian or Mexican. Note: `sex` encoding differs — Lithuanian uses 0/1 (coding unknown from metadata), Mexican uses 1 = Male, 2 = Female. All other binary columns use 0 = absent/negative, 1 = present/positive. `conversion_outcome`: 1 = converted to MS, 0 = did not. The result is saved to `visuals/cross_population_union.csv` for Power BI.

In [7]:
q6 = """
SELECT 'Lithuanian' AS population,
       CAST(Sex AS INTEGER) AS sex,
       Age AS age,
       CASE WHEN TRIM("OGB + in CSF") IN ('0','1')
            THEN CAST(TRIM("OGB + in CSF") AS INTEGER) ELSE NULL END AS oligoclonal_bands,
       CASE WHEN TRIM(Periventricular) IN ('0','1')
            THEN CAST(TRIM(Periventricular) AS INTEGER) ELSE NULL END AS periventricular_mri,
       CASE WHEN TRIM("MRI lesion localisation: infratentorally") IN ('0','1')
            THEN CAST(TRIM("MRI lesion localisation: infratentorally") AS INTEGER) ELSE NULL END AS infratentorial_mri,
       CASE WHEN TRIM("MRI spinal lesions") IN ('0','1')
            THEN CAST(TRIM("MRI spinal lesions") AS INTEGER) ELSE NULL END AS spinal_cord_mri,
       CASE WHEN TRIM("VEP +") IN ('0','1')
            THEN CAST(TRIM("VEP +") AS INTEGER) ELSE NULL END AS vep,
       CASE WHEN TRIM("BAEP +") IN ('0','1')
            THEN CAST(TRIM("BAEP +") AS INTEGER) ELSE NULL END AS baep,
       CAST(MS AS INTEGER) AS conversion_outcome
FROM lithuanian
WHERE MS IS NOT NULL
UNION ALL
SELECT 'Mexican' AS population,
       CASE WHEN SUBSTR(TRIM(Gender), 1, 1) IN ('1','2')
            THEN CAST(SUBSTR(TRIM(Gender), 1, 1) AS INTEGER) ELSE NULL END AS sex,
       "Age (y)" AS age,
       CASE WHEN SUBSTR(TRIM("Oligoclonal bands"), 1, 1) IN ('0','1')
            THEN CAST(SUBSTR(TRIM("Oligoclonal bands"), 1, 1) AS INTEGER) ELSE NULL END AS oligoclonal_bands,
       CASE WHEN SUBSTR(TRIM("Periventricular MRI"), 1, 1) IN ('0','1')
            THEN CAST(SUBSTR(TRIM("Periventricular MRI"), 1, 1) AS INTEGER) ELSE NULL END AS periventricular_mri,
       CASE WHEN SUBSTR(TRIM("Infratentorial MRI"), 1, 1) IN ('0','1')
            THEN CAST(SUBSTR(TRIM("Infratentorial MRI"), 1, 1) AS INTEGER) ELSE NULL END AS infratentorial_mri,
       CASE WHEN SUBSTR(TRIM("Spinal cord MRI"), 1, 1) IN ('0','1')
            THEN CAST(SUBSTR(TRIM("Spinal cord MRI"), 1, 1) AS INTEGER) ELSE NULL END AS spinal_cord_mri,
       CASE WHEN SUBSTR(TRIM(VEP), 1, 1) IN ('0','1')
            THEN CAST(SUBSTR(TRIM(VEP), 1, 1) AS INTEGER) ELSE NULL END AS vep,
       CASE WHEN SUBSTR(TRIM(BAEP), 1, 1) IN ('0','1')
            THEN CAST(SUBSTR(TRIM(BAEP), 1, 1) AS INTEGER) ELSE NULL END AS baep,
       CASE WHEN SUBSTR(TRIM("group"), 1, 1) = '1' THEN 1 ELSE 0 END AS conversion_outcome
FROM mexican
WHERE "group" IS NOT NULL
ORDER BY population, conversion_outcome
"""
df_q6 = pd.read_sql(q6, conn)
print(f'Union shape: {df_q6.shape}')
display(df_q6.head(10))

os.makedirs('../visuals', exist_ok=True)
df_q6.to_csv('../visuals/cross_population_union.csv', index=False)
print('Saved: visuals/cross_population_union.csv')

Union shape: (413, 10)


,population,sex,age,oligoclonal_bands,periventricular_mri,infratentorial_mri,spinal_cord_mri,vep,baep,conversion_outcome
0,Lithuanian,0,56.0,0.0,1.0,0.0,NaN,0.0,0.0,0
1,Lithuanian,0,47.0,0.0,0.0,1.0,NaN,0.0,1.0,0
2,Lithuanian,0,43.0,NaN,1.0,0.0,NaN,1.0,0.0,0
3,Lithuanian,0,39.0,0.0,1.0,0.0,NaN,NaN,NaN,0
4,Lithuanian,0,61.0,1.0,0.0,0.0,0.0,1.0,0.0,0
5,Lithuanian,1,37.0,0.0,NaN,NaN,0.0,0.0,0.0,0
6,Lithuanian,0,42.0,0.0,0.0,0.0,NaN,0.0,1.0,0
7,Lithuanian,0,70.0,0.0,0.0,0.0,NaN,0.0,0.0,0
8,Lithuanian,0,54.0,0.0,0.0,0.0,NaN,1.0,1.0,0
9,Lithuanian,0,44.0,0.0,0.0,0.0,NaN,0.0,0.0,0


Saved: visuals/cross_population_union.csv


## Save All Queries to sql/queries.sql

Write all six query strings to a single SQL file with comment headers, suitable for direct use in DB Browser for SQLite or any other SQLite client.

In [8]:
query_labels = [
    ('Query 1: Converters vs non-converters and conversion rate per cohort', q1),
    ('Query 2: Average age by conversion outcome per cohort', q2),
    ('Query 3: Oligoclonal band positivity rate by conversion outcome per cohort', q3),
    ('Query 4: MRI lesion presence rates by conversion outcome per cohort', q4),
    ('Query 5: VEP and BAEP positivity rates by conversion outcome per cohort', q5),
    ('Query 6: Cross-population union of shared comparable variables', q6),
]

with open('../sql/queries.sql', 'w', encoding='utf-8') as f:
    for label, sql in query_labels:
        f.write(f'-- {label}\n')
        f.write(sql.strip())
        f.write('\n\n\n')

conn.close()
print('Saved: sql/queries.sql')
print('Database connection closed.')

Saved: sql/queries.sql
Database connection closed.
